In [14]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([

    html.Center( 
        html.A( 
            href='https://www.snhu.edu', 
            children=[ 
                html.Img( 
                    src='data:image/png;base64,{}'.format(encoded_image.decode()), 
                    style={'width': '300px'} 
                ) 
            ], 
            target='_blank' 
        ) 
    ),

    html.Center(
        html.B(
            html.H1('CS-340 Dashboard')
        )
    ),

    html.Hr(),

    # dropdown filter
    html.Div([
        dcc.Dropdown(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster Rescue or Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            clearable=False
        )
    ]),

    html.Hr(),

    # data table 
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],
        data=df.to_dict('records'),
        
        # user friendly data view from module 6
        row_selectable="single",
        selected_rows=[0],

        filter_action="native",
        sort_action="native",
        sort_mode="multi",

        page_action="native",
        page_current=0,
        page_size=10
    ),

    html.Br(),
    html.Hr(),
    html.H1('Nicole'),

    # graph and map
    html.Div(
        className='row',
        style={'display': 'flex', 'width': '100%'},
        children=[

            html.Div(
                id='graph-id',
                style={'width': '50%'}
            ),

            html.Div(
                id='map-id',
                style={'width': '50%'}
            )

        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    # reset to return to full database view
    if filter_type == 'reset':
        return df.to_dict('records')

    # start with dogs only
    filtered_df = df[df['animal_type'] == 'Dog'].copy()

    # water rescue
    if filter_type == 'water':

        filtered_df = filtered_df[
            (filtered_df['breed'].str.contains(
                'Labrador Retriever Mix|Chesapeake Bay Retriever|Newfoundland',
                case=False,
                na=False
            ))
            &
            (filtered_df['sex_upon_outcome'] == 'Intact Female')
            &
            (filtered_df['age_upon_outcome_in_weeks'].between(26, 156))
        ]

    # mountain/wilderness rescue
    elif filter_type == 'mountain':

        filtered_df = filtered_df[
            (filtered_df['breed'].str.contains(
                'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler',
                case=False,
                na=False
            ))
            &
            (filtered_df['sex_upon_outcome'] == 'Intact Male')
            &
            (filtered_df['age_upon_outcome_in_weeks'].between(26, 156))
        ]

    # disaster/individual tracking
    elif filter_type == 'disaster':

        filtered_df = filtered_df[
            (filtered_df['breed'].str.contains(
                'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler',
                case=False,
                na=False
            ))
            &
            (filtered_df['sex_upon_outcome'] == 'Intact Male')
            &
            (filtered_df['age_upon_outcome_in_weeks'].between(20, 300))
        ]

    print("FILTER:", filter_type)
    print("ROWS:", len(filtered_df))

    return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'data')
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return html.Div(
            "No animals match this rescue filter."
        )

    dff = pd.DataFrame(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return html.Div(
            "No animals match this rescue filter."
        )

    breed_counts = (
        dff['breed']
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.bar(
                breed_counts,
                x='breed',
                y='count',
                title='Top 10 Breeds at AAC',
                labels={
                    'breed': 'Breed',
                    'count': 'Number of Animals'
                }
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'derived_virtual_selected_rows')
)
def update_styles(selected_rows):

    if selected_rows is None or len(selected_rows) == 0:
        return []

    return [{
        'if': {
            'row_index': selected_rows[0]
        },
        'backgroundColor': '#D2F3FF'
    }]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return html.Div(
            "No animals match this rescue filter."
        )

    dff = pd.DataFrame(viewData)

    if dff.empty:
        return html.Div(
            "No animals match this rescue filter."
        )

    # use first row if nothing is selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # make sure  selected row still exists
    if row >= len(dff):
        row = 0

    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']

    if pd.isna(latitude) or pd.isna(longitude):
        return html.Div(
            "Location data unavailable for this animal."
        )

    return dl.Map(
        style={
            'width': '100%',
            'height': '500px'
        },
        center=[latitude, longitude],
        zoom=10,
        children=[
            dl.TileLayer(),

            dl.Marker(
                position=[
                    latitude,
                    longitude
                ],
                children=[
                    dl.Tooltip(
                        dff.iloc[row]['breed']
                    ),
                    dl.Popup([
                        html.H4("Animal Name"),
                        html.P(
                            dff.iloc[row]['name']
                        )
                    ])
                ]
            )
        ]
    )

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://controlnumber-leonidindia-3000.codio.io/proxy/8050/
FILTER: mountain
ROWS: 19
FILTER: disaster
ROWS: 28
FILTER: water
ROWS: 17
